# Практическая работа №3


In [1]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from pyvis.network import Network

In [2]:
df = pd.read_csv("FinFraud_Labelled.csv", sep="|", header=None)
df.head()

/var/folders/9g/70hf9htx7jd7n3wvgghct5p40000gs/T/ipykernel_72526/3811695291.py:1: DtypeWarning: Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("FinFraud_Labelled.csv", sep="|", header=None)


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,N-RegC2C,PN_EU_3_4,PN_EU_0_883,EUAcc3_4,EUAcc0_883,68897.74,Ind,SU,1.000000e+08,9.993041e+07,...,NaN,01/06/2011 00:09:01,01/06/2011 00:09:01,EUAcc3_4,NaN,NaN,C2C201161.099,01/06/2011 00:09:01,EU,EU
2,N-RegC2C,PN_EU_1_139,PN_EU_0_754,EUAcc1_139,EUAcc0_754,68945.47,Ind,SU,1.000000e+08,9.993037e+07,...,NaN,01/06/2011 00:15:23,01/06/2011 00:15:23,EUAcc1_139,NaN,NaN,C2C201161.01515,01/06/2011 00:15:23,EU,EU
3,N-RegDep,PN_Ret2,PN_EU_3_17,RAcc2,EUAcc3_17,9715.41,Dt,SU,1.000000e+09,9.999903e+08,...,NaN,01/06/2011 00:22:07,01/06/2011 00:22:07,RAcc2,NaN,NaN,Dp201161.02222,01/06/2011 00:22:07,RET,EU
4,N-RegDep,PN_Ret1,PN_EU_0_266,RAcc1,EUAcc0_266,79303.74,Dt,SU,1.000000e+09,9.999207e+08,...,NaN,01/06/2011 00:22:35,01/06/2011 00:22:35,RAcc1,NaN,NaN,Dp201161.02222,01/06/2011 00:22:35,RET,EU


In [3]:
sender_col = df.columns[1]
receiver_col = df.columns[2]
amount_col = df.columns[5]

df_small = df[[sender_col, receiver_col, amount_col]].head(500)
df_small.head()

,1,2,5
0,NaN,NaN,NaN
1,PN_EU_3_4,PN_EU_0_883,68897.74
2,PN_EU_1_139,PN_EU_0_754,68945.47
3,PN_Ret2,PN_EU_3_17,9715.41
4,PN_Ret1,PN_EU_0_266,79303.74


In [4]:
G = nx.from_pandas_edgelist(
    df_small,
    source=sender_col,
    target=receiver_col,
    edge_attr=amount_col,
    create_using=nx.DiGraph()
)

print('Количество узлов:', G.number_of_nodes())
print('Количество ребер:', G.number_of_edges())

Количество узлов: 480
Количество ребер: 469


### Plotly

In [7]:
pos = nx.spring_layout(G, k=0.15)

edge_x = []
edge_y = []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5),
    hoverinfo='none',
    mode='lines')

node_x = []
node_y = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    marker=dict(size=8)
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.show()

## График Plotly
<img src="plotly.png" width="400">

### PyVis

In [8]:
net = Network(notebook=True, directed=True, cdn_resources="in_line")

for node in G.nodes():
    net.add_node(str(node))

for u, v, data in G.edges(data=True):
    net.add_edge(str(u), str(v))

net.show("graph.html")


graph.html


## Изображения из файла graph.html

<img src="image_1.png" width="400">
<img src="image_2.png" width="400">
<img src="image_3.png" width="400">


### Вывод


В результате визуализации графа транзакций были выявлены характерные структуры взаимодействия между пользователями. На графе наблюдаются узлы с большим количеством связей, что может свидетельствовать о централизованном распределении денежных средств. Также выделяются плотные кластеры аккаунтов, активно совершающих транзакции между собой, что может указывать на координированные действия и возможные мошеннические схемы. Наличие промежуточных узлов, через которые проходит большое количество переводов, может свидетельствовать о попытке скрыть источник средств.